<a href="https://colab.research.google.com/github/AliMehdii/Memoire-2023/blob/master/Copy_of_Version_01_Modeling_Colab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2

# import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
import imutils 

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.callbacks import TensorBoard



In [ ]:
tf.config.list_physical_devices('GPU')

[]

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)
    google_drive_prefix = "/content/drive/My Drive"
    data_prefix = "{}/mnist/".format(google_drive_prefix)
except ModuleNotFoundError: 
    data_prefix = "data/"

Mounted at /content/drive/


In [ ]:
!pip install wandb

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 KB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 KB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 KB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 KB 14.0 MB/s eta 0:00:00
  Created wheel for pathtools: filename=pathtools-0.1.2-py3-none-any.whl size=8806 sha256=657c8f04b95c966f0450ef3aab80d65888dc238ed28862c4e5adde337de0fe7f
  Stored in directory: /root/.cache/pip/wheels/4c/8e/7e/72fbc243e1aeecae64a96875432e70d4e92f3d2d18123be004
Successfully built pathtools
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3


In [ ]:
import wandb
wandb.login()

ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 

··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint

In [ ]:
# Start a run, tracking hyperparameters
wandb.init(
    # set the wandb project where this run will be logged
    project="Mermoire_2023_Version_01",

    # track hyperparameters and run metadata with wandb.config
    config={
        "dropout": 0.5,
        "dropout_2": 0.5,
        "activation": "softmax",
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 40,
        "batch_size": 32
    }
)

batch/accuracy,▁▁▃▅▆▆▇▆▇▇▇█▇█▇██▇██████████████████████
batch/batch_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▇▅▄▃▃▂▃▂▂▂▂▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▃▅▆▆▆▇▇▇▇▇▇██▇█████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▄▃▃▃▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▆▆▂▄▇▄▅▇▄▆▆▄▅▆▆▇▆▄▇▆▇▅▇▇▅▇▇▆█▇
epoch/val_loss,▆▅▃▃▅▂▁▃▃▁▄▃▂▄▄▃▄▃▅▇▂▄▃█▅▃█▄▄▆▄▄
batch/accuracy,0.98975


In [ ]:
config = wandb.config

In [ ]:
train_set = '/content/drive/My Drive/Cropped_Image_Sets/train'
val_set = '/content/drive/My Drive/Cropped_Image_Sets/val'
test_set = '/content/drive/My Drive/Cropped_Image_Sets/test'
model_dir ="/content/drive/My Drive/Models/RadImageNet-ResNet50_notop.h5"
IMAGE_SIZE = 128

In [ ]:
def init_data(train_dir: str, valid_dir: str, test_dir: str) -> list:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    
    train_data = train_datagen.flow_from_directory(
        directory=train_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    valid_data = valid_datagen.flow_from_directory(
        directory=valid_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    
    test_data = valid_datagen.flow_from_directory(
        directory=test_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    
    return train_data, valid_data, test_data

In [ ]:
train_data, valid_data, test_data = init_data(train_dir=train_set, valid_dir=val_set, test_dir=test_set)

Found 2144 images belonging to 3 classes.
Found 458 images belonging to 3 classes.
Found 472 images belonging to 3 classes.


In [ ]:
model_name = "My_model"

TensorBoard = TensorBoard(log_dir="logs\\{}".format(model_name))

In [ ]:
def build_transfer_learning_model(base_model):
    # `base_model` stands for the pretrained model
    # We want to use the learned weights, and to do so we must freeze them
    for layer in base_model.layers:
        layer.trainable = False
        
    # Declare a sequential model that combines the base model with custom layers
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(rate=config.dropout),
        tf.keras.layers.GlobalMaxPooling2D,
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(rate=config.dropout_2),
        tf.keras.layers.Dense(units=3, activation=config.activation)
    ])
    
    # Compile the model
    model.compile(
        loss=config.loss,
        optimizer=config.optimizer,
        metrics=[config.metric]
    )
    
    return model

In [ ]:
rad_model = build_transfer_learning_model(
    base_model=ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
)

In [ ]:
rad_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 2048)              23587712  
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dropout (Dropout)           (None, 2048)              0         
                                                                 
 batch_normalization (BatchN  (None, 2048)             8192      
 ormalization)                                                   
                                                                 
 dense (Dense)               (None, 128)               262272    
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                        

In [ ]:
# Train the model for 10 epochs
rad_hist = rad_model.fit(
    train_data,
    validation_data=valid_data,
    epochs=config.epoch,
    callbacks= [TensorBoard,
                WandbMetricsLogger(log_freq=5),
                WandbModelCheckpoint("models"),
                ]
)
wandb.finish()

Epoch 1/40
67/67 [==============================] - ETA: 0s - loss: 1.3454 - accuracy: 0.4636

wandb: Adding directory to artifact (./models)... Done. 0.5s


67/67 [==============================] - 781s 11s/step - loss: 1.3454 - accuracy: 0.4636 - val_loss: 0.9930 - val_accuracy: 0.5502
Epoch 2/40
67/67 [==============================] - ETA: 0s - loss: 0.9025 - accuracy: 0.6189

wandb: Adding directory to artifact (./models)... Done. 0.5s


67/67 [==============================] - 222s 3s/step - loss: 0.9025 - accuracy: 0.6189 - val_loss: 0.9053 - val_accuracy: 0.5873
Epoch 3/40
67/67 [==============================] - ETA: 0s - loss: 0.6497 - accuracy: 0.7374

...

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 216s 3s/step - loss: 0.0511 - accuracy: 0.9841 - val_loss: 0.7939 - val_accuracy: 0.7664
Epoch 32/40
67/67 [==============================] - ETA: 0s - loss: 0.0468 - accuracy: 0.9832

wandb: Adding directory to artifact (./models)... Done. 0.5s


67/67 [==============================] - 215s 3s/step - loss: 0.0468 - accuracy: 0.9832 - val_loss: 0.8644 - val_accuracy: 0.7293
Epoch 33/40
65/67 [============================>.] - ETA: 4s - loss: 0.0359 - accuracy: 0.9904

KeyboardInterrupt: ignored